# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Direction: Freestyle — "Confound or Cause?"** *(per `docs/ml-intern-dataset-and-lane-guide.md` section 9: freestyle needs no approval, same rigor bar as the four core lanes.)*

**My own question:** `content_type` and `model_used` are nearly the same variable in this dataset, and `content_age_days` differs by ~270 days across models. So before anyone claims "Model X's content performs better," I need to check whether that's a real model effect or just content type and age wearing a disguise.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: Classification, used diagnostically (signal attribution), not for deployment.**

I'm not trying to rank pages or ship a decision engine here. I'm training the *same* classifier twice on the *same* label, once with only the confounding variables (`content_type`, `content_age_days`) and once with `model_used` added on top. If adding `model_used` barely moves the model, that's evidence its apparent effect on decline is mostly a confound. If it moves the model a lot, that's evidence of a real, independent effect. The classifier is a measuring instrument here, not the product.

In [ ]:
lane = "Freestyle: Confound or Cause?"
task_type = "Classification, used diagnostically to compare two feature sets (with vs. without model_used)"
decision = "Are current model-performance claims trustworthy enough to act on at all?"
print(f"Lane: {lane}\nTask type: {task_type}\nDecision this answers: {decision}")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label = (trend_direction == "down")`.**

Same proxy caveat as always: this is a defined-rule label derived from `trend_pct`, not an independently observed future outcome, so `trend_direction`/`trend_pct` stay out of the feature set entirely (that's the Week 2 leakage trap). For this diagnostic, the label itself doesn't need to be perfect — I'm not trying to predict decline well, I'm testing whether one specific feature (`model_used`) adds anything once its confounds are already in the model.

In [ ]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Drop rows with no real model recorded (NaN / "unknown") -- can't test a confound
# against a value that isn't actually known.
sub = df[df["model_used"].notna() & (df["model_used"] != "unknown")].copy()
print("Rows with a known model_used:", sub.shape[0], "of", df.shape[0])
print("Declining rate in this slice:", round(sub["is_declining_label"].mean(), 3))

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**AUC lift from adding `model_used` on top of `content_type` + `content_age_days`.**

I fit Model A on the confounds only, Model B on the confounds plus `model_used`, both on the same held-out test split. If Model B's AUC is barely above Model A's, `model_used` isn't adding independent signal — the raw association was riding on the confounds. This is the one number that actually answers the decision: can FlyRank trust a model-performance claim, or is it measuring content type and age instead?

In [ ]:
X_confounds = pd.get_dummies(sub[["content_type", "content_age_days"]], columns=["content_type"])
X_full = pd.get_dummies(sub[["content_type", "content_age_days", "model_used"]],
                        columns=["content_type", "model_used"])
y = sub["is_declining_label"].values

Xa_tr, Xa_te, y_tr, y_te = train_test_split(X_confounds, y, test_size=0.3, random_state=42, stratify=y)
Xb_tr, Xb_te, _, _      = train_test_split(X_full,      y, test_size=0.3, random_state=42, stratify=y)

tree_a = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(Xa_tr, y_tr)
tree_b = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(Xb_tr, y_tr)

auc_a = roc_auc_score(y_te, tree_a.predict_proba(Xa_te)[:, 1])
auc_b = roc_auc_score(y_te, tree_b.predict_proba(Xb_te)[:, 1])

print(f"AUC, confounds only (content_type + content_age_days): {auc_a:.4f}")
print(f"AUC, confounds + model_used:                           {auc_b:.4f}")
print(f"Lift from adding model_used: {auc_b - auc_a:+.4f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item, restricted to rows with a known `model_used`.** Before touching any model, here's the confound itself, in two tables: how tightly `content_type` is tied to `model_used`, and how differently aged each model's content is.

In [ ]:
print("content_type share within each model_used (rows sum to 1.0 per row):")
print(pd.crosstab(sub["model_used"], sub["content_type"], normalize="index").round(3))
print()
print("Average content_age_days by model_used:")
print(sub.groupby("model_used")["content_age_days"].mean().round(1))

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule — a raw groupby like "compare decline rate by model_used" — can't hold anything else constant. It would report whatever mix of model, content type, and age happens to be in the data, with no way to separate them. Isolating "does `model_used` matter *independent of* content type and age" requires comparing two models fit on the same rows and the same held-out split, one with the confounds only and one with `model_used` added — that comparison, not a single number, is what a rule can't produce.

Here's what that comparison actually shows:

In [ ]:
imp = pd.Series(tree_b.feature_importances_, index=X_full.columns).sort_values(ascending=False)
print("Feature importances in the full model (confounds + model_used):")
print(imp.round(4))
print()
print(f"AUC lift from adding model_used: {auc_b - auc_a:+.4f}")
print("-> content_age_days and content_type dominate; model_used columns contribute almost")
print("   nothing on top. The apparent 'model effect' is mostly confound, not cause.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.